# Experiments 84-85
Combination of hyperparameters that provided the best results.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v5i)***
    - Plus soil images
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. Mix of hyperparms:

        1. Shorter train cycle (2 hours w/ 30min steps)
        1. Short train cycle (30min)

        - **Loss gain implementation (>cls | <box)**
        - multi_scale | weight_decay | momentum | dropout
        - 9x augmentation *(synthetic data)*
        - Full Fine-Tuning *(No freeze)*
        - IOU/CONF optimization _(on valid)_

## Init

In [31]:
import os
import shutil
import fnmatch
import pickle

In [32]:
!pip install ultralytics

## Helper Functions

In [33]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [34]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [35]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [36]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [37]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [38]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Validation functions

In [4]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [40]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [41]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [3]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [1]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [21]:
!rm -rf /content/sample_data

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px.aug.v1
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  3.5m.v5i.yolov8.640px-2steps.aug2
3.5m.v3i.yolov8.640px_clahe	       best_e26.pt
3.5m.v3i.yolov8.640px.soil_aug	       best_e50.pt
3.5m.v4i.yolov8.640px		       Inference
3.5m.v4i.yolov8.640px_209	       models
3.5m.v4i.yolov8.640px.aug.v1	       optuna_yolov8_f1_study.db


In [46]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 16 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8.640px_209',
 '3.5m.v4i.yolov8_blended.640px.aug.v1',
 'best_e50.pt',
 '3.5m.v4i.yolov8.640px.aug.v1',
 '3.5m.v5i.yolov8.640px-2steps.aug2']

**For this experiments:** `3.5m.v5i.yolov8.640px-2steps.aug2`

In [48]:
choose_dataset = 16
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v5i.yolov8.640px-2steps.aug2


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [25]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [49]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml'

## Download model

In [50]:
from ultralytics import YOLO

In [51]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

# Finetuning

### Optimization

In [69]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [70]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [14]:
!nvidia-smi

Thu May 15 10:17:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
!yolo version

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.3.134


-----
## Experiment 84
### *YOLOv8 Mid | Longer train cycle*

### Train

In [75]:
# Garbage collection
import gc
for i in range(20):
  torch.cuda.empty_cache()
  gc.collect()

In [56]:
# Set's maximum training time (in hours)
time: float = 2 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=50,
    time = time,
    multi_scale=True,
    weight_decay=0.0015,
    dropout=0.05,
    #momentum=0.99,
    cls=1, # Higher than 0.5 (default)
    box=5, # Lower than 7.5 (default)
    # dfl 1.5 (default)
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.0, plots=Tru

train: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.31G reserved, 0.30G allocated, 14.13G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.676         35.13         221.7        (1, 3, 640, 640)                    list
    25856899       158.1         2.187         35.56         111.9        (2, 3, 640, 640)                    list
    25856899       316.3         3.049         53.25         122.1        (4, 3, 640, 640)                    list
    25856899       632.5         4.626         80.05         152.1        (8, 3, 640, 640)                    list
    25856899        1265         7.692         152.1         314.7       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.35G/14.74G (57%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1837.9±697.5 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      71/71      12.4G      1.138      1.805      1.284        259        608:  57%|█████▋    | 92/161 [00:56<00:42,  1.63it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.20it/s]

                   all        108       3467      0.606      0.529      0.533      0.185



71 epochs completed in 2.001 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.0MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.71s/it]


                   all        108       3467      0.568      0.537      0.536      0.194
Speed: 0.2ms preprocess, 13.5ms inference, 0.0ms loss, 8.8ms postprocess per image
Results saved to runs/detect/train


In [77]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c337b7ca590>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [78]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=2,
          patience=50,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.05,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          io

In [79]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Save results

In [80]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


### Validation

In [81]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [82]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1476.3±251.7 MB/s, size: 87.0 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.49s/it]


                   all        108       3467      0.602      0.564      0.573      0.226
Speed: 8.0ms preprocess, 22.6ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [83]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [84]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [85]:
matrix = gimme_metrics(results)

Total objects detected: 4592.0
Confusion matrix:
['46.34%', '24.50%']
['29.16%', '0.00%']


### Save results

In [86]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


### Metrics

In [87]:
matrix

[[2128.0, 1125.0], [1339.0, 0.0]]

In [88]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4592.0

Confusion matrix:
[ 46.34% , 24.50% ]
[ 29.16% , 0.00% ]

Metrics:
- Accuracy: 0.463
- Precision: 0.654
- Recall: 0.614
- F1 Score: 0.633
- F½ Score: 0.646
- G-mean: 0.634


-----
## Experiment 85
### *YOLOv8 Mid | Shorter train cycle*

### Train

In [ ]:
# Garbage collection
import gc
for i in range(20):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Set's maximum training time (in hours)
time: float = 0.5 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=10,
    time = time,
    multi_scale=True,
    weight_decay=0.0015,
    dropout=0.05,
    #momentum=0.99,
    cls=1, # Higher than 0.5 (default)
    box=5, # Lower than 7.5 (default)
    # dfl 1.5 (default)
)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=T

train: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.42G reserved, 0.38G allocated, 13.94G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.674         34.95         42.03        (1, 3, 640, 640)                    list
    25856899       158.1         2.189         42.67         74.15        (2, 3, 640, 640)                    list
    25856899       316.3         3.070         44.94         81.69        (4, 3, 640, 640)                    list
    25856899       632.5         4.672         82.62         140.1        (8, 3, 640, 640)                    list
    25856899        1265         7.697         155.8         272.8       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 16 for CUDA:0 8.56G/14.74G (58%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1826.0±489.4 MB/s, size: 74.4 KB)


train: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/train/labels.cache... 2570 images, 141 backgrounds, 0 corrupt: 100%|██████████| 2570/2570 [00:00<?, ?it/s]

...

      17/17      12.7G      1.244      2.138       1.31        275        576:  91%|█████████ | 146/161 [01:36<00:09,  1.52it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.26it/s]

                   all        108       3467      0.596      0.536      0.549      0.202



17 epochs completed in 0.501 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.0MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.73s/it]


                   all        108       3467      0.585      0.535       0.55      0.202
Speed: 0.3ms preprocess, 12.5ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/train2


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c342ffa1290>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml',
          epochs=500,
          time=0.5,
          patience=10,
          batch=16,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.05,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
         

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save2/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save2/


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.134 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2163.2±778.1 MB/s, size: 78.3 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.68s/it]


                   all        108       3467      0.621      0.539      0.572       0.23
Speed: 5.9ms preprocess, 22.4ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4418.0
Confusion matrix:
['46.56%', '21.53%']
['31.91%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save2/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save2/


### Metrics

In [ ]:
matrix

[[2057.0, 951.0], [1410.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4418.0

Confusion matrix:
[ 46.56% , 21.53% ]
[ 31.91% , 0.00% ]

Metrics:
- Accuracy: 0.466
- Precision: 0.684
- Recall: 0.593
- F1 Score: 0.635
- F½ Score: 0.664
- G-mean: 0.637
